# Questions → Relations & Entities (Knowledge Graph Extraction)

- Locates the project root by walking up the directory tree until a `DATA` folder is found, then sets the target `CATEGORY` and the Mistral API key.
- Loads the disjoint MCQ subset CSV for that category (question + correct answer + 3 distractors per row).
- **RELATIONS**: for each article (rows grouped by `article_id`), queries Mistral with a system prompt asking it to reformulate each question into predicate(s) usable as graph edges, then deduplicates them per article.
- **ENTITIES**: for each CSV row, queries Mistral to extract the named entities (people, places, works, dates, cultural terms...) from the answer, distractors and question, then aggregates them per `article_id`.
- Both stages run in a two-level thread pool (articles × texts) with per-article timeouts, cooperative cancellation and up to `MAX_ROUNDS` retry passes (longer timeout / fewer workers each pass); results are written as JSON.

In [ ]:
from pathlib import Path

# Walk up the tree until the DATA folder is found
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)

In [ ]:
CATEGORY = "literatura"
API_KEY = ""

# RELATIONS

Extraction of the predicates (graph edges) implied by each question.
Rows are grouped by `article_id`, and every question/answer/distractor of the
article is sent to the model; the relations are then deduplicated per article.

In [ ]:
import os
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout

from mistralai import Mistral
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CSV_PATH = str(DATA / "SUBSETS_with_relations" / "ARTICLES_SUBSETS_ES" / f"{CATEGORY}_articles_es_disjoint.csv")
OUTPUT_JSON = str(DATA / "SUBSETS_with_relations" / "RELATIONS_EXTRAITES" / f"relations_extraites_{CATEGORY}_es.json")

MODEL = "mistral-small-2506"
MODEL_title = MODEL.split("/")[-1].replace("-", "_")

if not API_KEY:
    raise ValueError("The MISTRAL_API_KEY environment variable is not set.")

client = Mistral(api_key=API_KEY)

MAX_WORKERS_ARTICLES = 8   # number of articles processed in parallel
MAX_WORKERS_TEXTES = 5     # number of texts processed in parallel WITHIN one article

# --- New robustness parameters ---------------------------------------------
ARTICLE_TIMEOUT = 10       # max seconds per article (pass 1)
REQUEST_TIMEOUT = 10       # max seconds per API call
MAX_ROUNDS = 4             # total number of passes (1 normal + 3 retries)
TIMEOUT_GROWTH = 3.0       # timeout multiplier applied at each pass
RETRY_SLEEP = 5            # pause between two passes

# ---------------------------------------------------------------------------
# System prompt: relation extraction for a knowledge graph
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """Eres un experto en extracción de relaciones para construir grafos de conocimiento.

Tu tarea: dada una PREGUNTA, identifica la(s) RELACIÓN(ES) que permitiría(n) encontrar la respuesta en un texto. La relación es un predicado (normalmente un verbo o locución verbal) que conecta un sujeto (src) con un objeto (dst).

REGLAS IMPORTANTES:
- NO extraigas las relaciones literalmente de la pregunta. Debes REFORMULAR.
- Identifica con precisión el predicado que, en un texto relevante, conectaría 
  la entidad conocida con la respuesta buscada.
- Puedes proponer varias relaciones si la pregunta lo requiere.
- La relación debe estar en forma afirmativa (no interrogativa).

Ejemplo:
Para la pregunta: "¿Qué hallazgo arqueológico realizó Selene Velázquez en el Templo de Nuestra Señora de los Dolores de Monterrey?"
La relación sería: "realizó el hallazgo de"
Esto permite construir, en un texto relevante, el triple:
(src="Selene Velázquez", relation="'realizó el hallazgo de ", dst="más de 2,200 ollas de barro")

Responde ÚNICAMENTE con un objeto JSON con esta estructura exacta:
{"relations": ["relacion1", "relacion2", ...]}

No añadas texto adicional, explicaciones ni comentarios."""

# ---------------------------------------------------------------------------
# Cooperative cancellation: prevents "zombie" threads from an abandoned pass
# from keeping on hitting the API during the next pass.
# ---------------------------------------------------------------------------
_cancel_event = threading.Event()


def _chat_complete(messages, timeout_s=REQUEST_TIMEOUT):
    """Mistral call with a network timeout when the SDK supports it."""
    try:
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
            timeout_ms=int(timeout_s * 1000),
        )
    except TypeError:
        # Older SDK: no timeout_ms parameter
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
        )


# ---------------------------------------------------------------------------
# Extraction functions
# ---------------------------------------------------------------------------
def extraire_relations(texte: str, max_retries: int = 3) -> list:
    """Query Mistral to extract the relations of a text."""
    if not isinstance(texte, str) or not texte.strip():
        return []

    content = ""
    for attempt in range(max_retries):
        # Bail out early if the current pass has been abandoned
        if _cancel_event.is_set():
            return []
        try:
            response = _chat_complete([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Texto : {texte}"},
            ])
            content = response.choices[0].message.content
            data = json.loads(content)
            relations = data.get("relations", [])
            if isinstance(relations, str):
                relations = [relations]
            return [r.strip() for r in relations if r and r.strip()]
        except json.JSONDecodeError:
            print(f"  [!] Non-JSON response (attempt {attempt+1}): {str(content)[:200]}")
        except Exception as e:
            print(f"  [!] API error (attempt {attempt+1}): {e}")
            time.sleep(2 * (attempt + 1))  # backoff

    return []  # failure after all attempts


def traiter_article(article_id, textes):
    """Process all texts of an article in parallel and return (id, unique relations)."""
    n = len(textes)
    relations_par_texte = [None] * n
    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS_TEXTES, max(n, 1))) as ex:
        futures = {ex.submit(extraire_relations, t): idx for idx, t in enumerate(textes)}
        for fut in as_completed(futures):
            idx = futures[fut]
            relations_par_texte[idx] = fut.result()

    # Flatten then deduplicate while preserving order
    flat = [r for sub in relations_par_texte if sub for r in sub]
    seen, uniques = set(), []
    for r in flat:
        if r not in seen:
            seen.add(r)
            uniques.append(r)
    return article_id, uniques


# ---------------------------------------------------------------------------
# Running a pass with a per-article timeout
# ---------------------------------------------------------------------------
def _executer_passe(taches, resultats, max_workers, timeout_article, label=""):
    """
    Runs `taches` = [(article_id, textes), ...].
    Fills `resultats` for the ones completing within the time limit.
    Returns the list of failed tasks (timeout or exception).
    """
    if not taches:
        return []

    _cancel_event.clear()
    nb = len(taches)
    en_echec = []

    executor = ThreadPoolExecutor(max_workers=max_workers)
    try:
        futures = {executor.submit(traiter_article, aid, txt): (aid, txt)
                   for aid, txt in taches}

        deadline_global = time.time() + timeout_article * (nb / max(max_workers, 1) + 2)

        done_count = 0
        for fut in as_completed(futures, timeout=max(0.1, deadline_global - time.time())) \
                if False else _as_completed_safe(futures, deadline_global):
            aid, txt = futures[fut]
            done_count += 1
            try:
                article_id, uniques = fut.result(timeout=timeout_article)
                print(f"{label}[{done_count}/{nb}] article_id={article_id} -> {len(uniques)} relations")
                print(f"   => unique relations: {uniques}\n")
                resultats[article_id] = uniques
            except FutTimeout:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ⏱️ TIMEOUT (>{timeout_article}s) -> postponed\n")
                en_echec.append((aid, txt))
            except Exception as e:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ❌ ERROR: {e} -> postponed\n")
                en_echec.append((aid, txt))

        # Futures that never completed (stuck) => postponed
        for fut, (aid, txt) in futures.items():
            if not fut.done():
                fut.cancel()
                print(f"{label}article_id={aid} ⏱️ STUCK -> postponed\n")
                en_echec.append((aid, txt))
    finally:
        # We do not wait for zombie threads: we just ask them to stop
        _cancel_event.set()
        executor.shutdown(wait=False)

    return en_echec


def _as_completed_safe(futures, deadline_global):
    """as_completed bounded by a global deadline, without raising an exception."""
    pending = set(futures)
    while pending:
        restant = deadline_global - time.time()
        if restant <= 0:
            return
        try:
            for fut in as_completed(list(pending), timeout=restant):
                pending.discard(fut)
                yield fut
            return
        except FutTimeout:
            return


# ---------------------------------------------------------------------------
# Main function (parallelized + retry of stuck articles)
# ---------------------------------------------------------------------------
def main():
    df = pd.read_csv(CSV_PATH)

    required_cols = ["article_id", "question", "answer", "distractor1", "distractor2", "distractor3"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Column '{col}' not found. Available columns: {list(df.columns)}"
            )

    print(f"{len(df)} questions loaded from the CSV.\n")

    # Grouping by article_id (one article = several questions)
    taches = []
    for article_id, group in df.groupby("article_id"):
        textes = []
        for row in group.itertuples(index=False):
            textes.extend([
                row.question,
                row.answer,
                row.distractor1,
                row.distractor2,
                row.distractor3,
            ])
        taches.append((article_id, textes))

    nb_articles = len(taches)
    print(f"{nb_articles} articles (grouped) to process.\n")

    resultats = {}          # {article_id: [relations...]}
    restantes = taches
    timeout = ARTICLE_TIMEOUT
    workers = MAX_WORKERS_ARTICLES

    for round_idx in range(1, MAX_ROUNDS + 1):
        if not restantes:
            break
        label = "" if round_idx == 1 else f"(retry {round_idx-1}) "
        if round_idx > 1:
            print(f"\n🔁 Pass {round_idx}: {len(restantes)} article(s) to retry "
                  f"(timeout={timeout:.0f}s, workers={workers})\n")
            time.sleep(RETRY_SLEEP)
        restantes = _executer_passe(restantes, resultats, workers, timeout, label)
        # Next pass: more time, less parallelism
        timeout = timeout * TIMEOUT_GROWTH
        workers = max(1, workers // 2)

    # Definitively failed articles -> empty list (homogeneous output)
    for aid, _ in restantes:
        print(f"⚠️  article_id={aid}: definitive failure after {MAX_ROUNDS} passes -> []")
        resultats[aid] = []

    # Reorder following the CSV article order (optional, more readable output)
    resultats = {aid: resultats.get(aid, []) for aid, _ in taches}

    os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(resultats, f, ensure_ascii=False, indent=2)

    ok = sum(1 for v in resultats.values() if v)
    print(f"\n✅ Done. {ok}/{nb_articles} articles with relations.")
    print(f"Results saved to: {OUTPUT_JSON}")

    return resultats


if __name__ == "__main__":
    resultats = main()

# ENTITIES

Named-entity extraction (people, places, works, dates, cultural terms...) from
the answer, the three distractors and the question of every CSV row.
Entities are aggregated and deduplicated per `article_id`.

In [ ]:
import os
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout

from mistralai import Mistral
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CSV_PATH = str(DATA / "SUBSETS_with_relations" / "ARTICLES_SUBSETS_ES" / f"{CATEGORY}_articles_es_disjoint.csv")
OUTPUT_ENTITIES_JSON = str(DATA / "SUBSETS_with_relations" / "ENTITES_EXTRAITES" / f"entites_extraites_{CATEGORY}_es.json")

MODEL = "mistral-small-2506"
MODEL_title = MODEL.split("/")[-1].replace("-", "_")

if not API_KEY:
    raise ValueError("The MISTRAL_API_KEY environment variable is not set.")

client = Mistral(api_key=API_KEY)

MAX_WORKERS = 8            # number of articles processed in parallel

# --- New robustness parameters ---------------------------------------------
ARTICLE_TIMEOUT_E = 10     # max seconds per row/article (pass 1)
REQUEST_TIMEOUT_E = 10     # max seconds per API call
MAX_ROUNDS_E = 4
TIMEOUT_GROWTH_E = 3.0
RETRY_SLEEP_E = 5

# ---------------------------------------------------------------------------
# System prompt: named-entity extraction
# ---------------------------------------------------------------------------
SYSTEM_PROMPT_ENTITIES = """Eres un experto en extracción de entidades nombradas para construir grafos de conocimiento.

Tu tarea: dado un TEXTO (respuesta correcta o distractores de una pregunta de comprensión), extrae las ENTIDADES NOMBRADAS relevantes. Considera como entidades: personas, lugares, platos, ingredientes, celebraciones, instituciones, fechas, obras, términos culturales, etc.

REGLAS IMPORTANTES:
- Extrae ÚNICAMENTE entidades concretas y relevantes para el contenido del texto.
- No incluyas pronombres, artículos ni palabras vacías.
- Si el texto es una lista, extrae cada elemento significativo por separado.
- Devuelve las entidades en minúsculas salvo nombres propios, manteniendo el formato original del nombre propio.

Responde ÚNICAMENTE con un objeto JSON con esta estructura exacta:
{"entities": ["entidad1", "entidad2", ...]}

No añadas texto adicional, explicaciones ni comentarios."""

_cancel_event_e = threading.Event()


def _chat_complete_e(messages, timeout_s=REQUEST_TIMEOUT_E):
    try:
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
            timeout_ms=int(timeout_s * 1000),
        )
    except TypeError:
        # Older SDK: no timeout_ms parameter
        return client.chat.complete(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"},
        )


# ---------------------------------------------------------------------------
# Extraction functions
# ---------------------------------------------------------------------------
def extraire_entites(texte: str, max_retries: int = 3) -> list:
    """Query Mistral to extract the entities of a text."""
    if not isinstance(texte, str) or not texte.strip():
        return []

    content = ""
    for attempt in range(max_retries):
        # Bail out early if the current pass has been abandoned
        if _cancel_event_e.is_set():
            return []
        try:
            response = _chat_complete_e([
                {"role": "system", "content": SYSTEM_PROMPT_ENTITIES},
                {"role": "user", "content": f"Texto : {texte}"},
            ])
            content = response.choices[0].message.content
            data = json.loads(content)
            entities = data.get("entities", [])
            if isinstance(entities, str):
                entities = [entities]
            return [e.strip() for e in entities if e and e.strip()]
        except json.JSONDecodeError:
            print(f"  [!] Non-JSON response (attempt {attempt+1}): {str(content)[:200]}")
        except Exception as e:
            print(f"  [!] API error (attempt {attempt+1}): {e}")
            time.sleep(2 * (attempt + 1))

    return []


def traiter_article_entites(article_id, textes):
    """Process the texts of an article in parallel and return (id, unique entities)."""
    entities_article = [None] * len(textes)
    with ThreadPoolExecutor(max_workers=max(1, len(textes))) as ex:
        futures = {ex.submit(extraire_entites, t): idx for idx, t in enumerate(textes)}
        for fut in as_completed(futures):
            idx = futures[fut]
            entities_article[idx] = fut.result()

    # Flatten then deduplicate while preserving order
    flat = [e for sub in entities_article if sub for e in sub]
    seen, uniques = set(), []
    for e in flat:
        if e not in seen:
            seen.add(e)
            uniques.append(e)
    return article_id, uniques


# ---------------------------------------------------------------------------
# Aggregation (same as the original: several rows -> same article_id)
# ---------------------------------------------------------------------------
def _agreger(resultats, article_id, uniques):
    if article_id in resultats:
        resultats[article_id].extend(uniques)
        # Merge while keeping the first occurrence order
        seen, merged = set(), []
        for e in resultats[article_id]:
            if e not in seen:
                seen.add(e)
                merged.append(e)
        resultats[article_id] = merged
    else:
        resultats[article_id] = list(uniques)


# ---------------------------------------------------------------------------
# Running a pass with a per-article timeout
# ---------------------------------------------------------------------------
def _as_completed_safe_e(futures, deadline_global):
    """as_completed bounded by a global deadline, without raising an exception."""
    pending = set(futures)
    while pending:
        restant = deadline_global - time.time()
        if restant <= 0:
            return
        try:
            for fut in as_completed(list(pending), timeout=restant):
                pending.discard(fut)
                yield fut
            return
        except FutTimeout:
            return


def _executer_passe_entites(taches, resultats, max_workers, timeout_article, label=""):
    if not taches:
        return []

    _cancel_event_e.clear()
    nb = len(taches)
    en_echec = []

    executor = ThreadPoolExecutor(max_workers=max_workers)
    try:
        futures = {executor.submit(traiter_article_entites, aid, txt): (aid, txt)
                   for aid, txt in taches}

        deadline_global = time.time() + timeout_article * (nb / max(max_workers, 1) + 2)

        done_count = 0
        for fut in _as_completed_safe_e(futures, deadline_global):
            aid, txt = futures[fut]
            done_count += 1
            try:
                article_id, uniques = fut.result(timeout=timeout_article)
                print(f"{label}[{done_count}/{nb}] article_id={article_id} -> {len(uniques)} entities")
                print(f"   => unique entities: {uniques}\n")
                _agreger(resultats, article_id, uniques)
            except FutTimeout:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ⏱️ TIMEOUT (>{timeout_article}s) -> postponed\n")
                en_echec.append((aid, txt))
            except Exception as e:
                print(f"{label}[{done_count}/{nb}] article_id={aid} ❌ ERROR: {e} -> postponed\n")
                en_echec.append((aid, txt))

        # Futures that never completed (stuck) => postponed
        for fut, (aid, txt) in futures.items():
            if not fut.done():
                fut.cancel()
                print(f"{label}article_id={aid} ⏱️ STUCK -> postponed\n")
                en_echec.append((aid, txt))
    finally:
        _cancel_event_e.set()
        executor.shutdown(wait=False)

    return en_echec


# ---------------------------------------------------------------------------
# Main function (parallelized + retry of stuck articles)
# ---------------------------------------------------------------------------
def main_entities():
    df = pd.read_csv(CSV_PATH)

    required_cols = ["article_id", "answer", "distractor1", "distractor2", "distractor3", "question"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Column '{col}' not found. Available columns: {list(df.columns)}"
            )

    print(f"{len(df)} articles loaded from the CSV.\n")

    # One task per CSV row (answer + distractors + question)
    taches = []
    for row in df.itertuples(index=False):
        textes = [
            row.answer,
            row.distractor1,
            row.distractor2,
            row.distractor3,
            row.question,
        ]
        taches.append((row.article_id, textes))

    nb_taches = len(taches)

    resultats = {}  # {article_id: [aggregated entities...]}
    restantes = taches
    timeout = ARTICLE_TIMEOUT_E
    workers = MAX_WORKERS

    for round_idx in range(1, MAX_ROUNDS_E + 1):
        if not restantes:
            break
        label = "" if round_idx == 1 else f"(retry {round_idx-1}) "
        if round_idx > 1:
            print(f"\n🔁 Pass {round_idx}: {len(restantes)} task(s) to retry "
                  f"(timeout={timeout:.0f}s, workers={workers})\n")
            time.sleep(RETRY_SLEEP_E)
        restantes = _executer_passe_entites(restantes, resultats, workers, timeout, label)
        # Next pass: more time, less parallelism
        timeout = timeout * TIMEOUT_GROWTH_E
        workers = max(1, workers // 2)

    for aid, _ in restantes:
        print(f"⚠️  article_id={aid}: definitive failure after {MAX_ROUNDS_E} passes -> []")
        if aid not in resultats:
            resultats[aid] = []

    # Stable order = CSV order
    ordre = []
    for aid, _ in taches:
        if aid not in ordre:
            ordre.append(aid)
    resultats = {aid: resultats.get(aid, []) for aid in ordre}

    os.makedirs(os.path.dirname(OUTPUT_ENTITIES_JSON), exist_ok=True)
    with open(OUTPUT_ENTITIES_JSON, "w", encoding="utf-8") as f:
        json.dump(resultats, f, ensure_ascii=False, indent=2)

    ok = sum(1 for v in resultats.values() if v)
    print(f"\n✅ Done. {ok}/{len(resultats)} articles with entities.")
    print(f"Entities saved to: {OUTPUT_ENTITIES_JSON}")

    return resultats


if __name__ == "__main__":
    resultats_entites = main_entities()